# Day 1 RAG Demo

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv(Path('..') / '.env')
api_key = os.getenv('GEMINI_API_KEY')
if not api_key:
    raise ValueError('GEMINI_API_KEY not found in day1/.env')
genai.configure(api_key=api_key)

docs_dir = Path('..') / 'sample_docs'
doc_paths = sorted(docs_dir.glob('*.txt'))
documents = [p.read_text(encoding='utf-8').strip() for p in doc_paths]
print('documents_loaded=', len(documents))

documents_loaded= 3


In [3]:
def split_text(text, size=120, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = end - overlap
    return [c for c in chunks if c]

chunks = []
metadatas = []
for i, doc in enumerate(documents):
    parts = split_text(doc)
    for j, part in enumerate(parts):
        chunks.append(part)
        metadatas.append({'doc': i + 1, 'chunk': j + 1})

print('chunks_count=', len(chunks))

chunks_count= 5


In [ ]:
import chromadb
from time import time

PREFERRED_EMBED_MODELS = ['models/text-embedding-004', 'models/embedding-001']

def select_embedding_model():
    for model_name in PREFERRED_EMBED_MODELS:
        try:
            genai.embed_content(model=model_name, content='ping')
            return model_name
        except Exception:
            continue
    raise RuntimeError(
        'No supported embedding model found. Check your API key/project access and available Gemini models.'
    )

EMBEDDING_MODEL = select_embedding_model()
print('embedding_model=', EMBEDDING_MODEL)

def embed_text(text):
    res = genai.embed_content(model=EMBEDDING_MODEL, content=text)
    return res['embedding']

embeddings = [embed_text(c) for c in chunks]
client = chromadb.Client()
collection_name = f'day1_rag_{int(time())}'
collection = client.create_collection(name=collection_name)
ids = [f'chunk_{i}' for i in range(len(chunks))]
collection.add(ids=ids, documents=chunks, metadatas=metadatas, embeddings=embeddings)
print('stored_chunks=', len(ids))

NotFound: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.

In [ ]:
def retrieve(query, n_results=3):
    q_emb = embed_text(query)
    return collection.query(query_embeddings=[q_emb], n_results=n_results)

question_1 = 'How much does the premium plan cost?'
retrieved_1 = retrieve(question_1, 3)
print('question_1=', question_1)
for i, d in enumerate(retrieved_1['documents'][0], start=1):
    print(f'{i}. {d}')

In [ ]:
model = genai.GenerativeModel('gemini-1.5-flash')

def answer_with_context(question, retrieved_docs):
    context = '\n\n'.join(retrieved_docs)
    prompt = (
        'Answer only from the context. If context is not relevant, say: No relevant information found.\n\n'
        f'Question: {question}\n\n'
        f'Context:\n{context}'
    )
    return model.generate_content(prompt).text

answer_1 = answer_with_context(question_1, retrieved_1['documents'][0])
print('answer_1=', answer_1)

question_2 = 'Who won the world cup in 1998?'
retrieved_2 = retrieve(question_2, 3)
answer_2 = answer_with_context(question_2, retrieved_2['documents'][0])
print('question_2=', question_2)
print('answer_2=', answer_2)